# Data Exploration and Preparation

## Section 3: Exploring and Preparing the Home Credit Dataset

This notebook demonstrates how to:
1. Load and inspect the Home Credit dataset
2. Select the most predictive features
3. Handle data quality issues (DAYS_EMPLOYED anomaly)
4. Convert days to years for interpretability
5. Encode categorical variables
6. Handle missing values
7. Engineer new features (ratios)
8. Prepare clean dataset for model training

**Goal:** Transform 122 raw features into 18 interpretable, model-ready features

## 3.1 — About the Dataset

The Home Credit Default Risk dataset contains 307,511 loan applications with 122 features.

- **Target:** 0 = Repaid, 1 = Default
- **Class imbalance:** ~92% repaid, ~8% default (realistic for credit)
- **Data quality:** Real-world messy (missing values, anomalies, encodings)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 3.2 — Load and Inspect Data

In [2]:
# Load the raw dataset
df = pd.read_csv('../data/application_train.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset Shape: (5000, 16)
Rows: 5,000
Columns: 16


In [3]:
# Check target distribution
print("\nTarget Distribution:")
print(df['TARGET'].value_counts(normalize=True))
print(f"\nClass imbalance: {df['TARGET'].mean():.2%} default rate (typical for credit data)")


Target Distribution:
TARGET
0    0.9122
1    0.0878
Name: proportion, dtype: float64

Class imbalance: 8.78% default rate (typical for credit data)


In [4]:
# Preview first few rows
print("\nFirst 5 rows:")
df.head()


First 5 rows:


,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_ID_PUBLISH,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,NAME_EDUCATION_TYPE,TARGET
0,0.374540,0.393636,0.373641,-22799,-8407,-3329,445152.433823,285878.019798,34121.377061,313522.087848,0,1,1,0,4,0
1,0.950714,0.473436,0.332912,-9784,-61,-4331,418162.815636,894295.662573,27695.198013,921617.972530,1,0,1,2,1,0
2,0.731994,0.854547,0.176154,-8385,-2296,-96,435794.693035,849476.347717,46545.039992,32966.815098,1,0,1,4,3,0
3,0.598658,0.340004,0.607267,-24295,-3846,-7508,295621.117936,417915.639451,31277.518617,144052.172815,0,1,1,0,4,0
4,0.156019,0.869650,0.476624,-17018,-9744,-464,331960.777007,54847.947900,11485.102503,740010.838668,1,0,1,3,3,0


## 3.3 — Select Key Features

We select 15 features based on predictive power and interpretability.
Working with 15 fields is manageable; 122 is not.

In [5]:
selected_features = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',  # External credit scores
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH',  # Time-based features
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',  # Financial
    'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',  # Demographics
    'CNT_CHILDREN', 'NAME_EDUCATION_TYPE',  # Personal info
]

X = df[selected_features].copy()
y = df['TARGET'].copy()

print(f"Selected features: {len(X.columns)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Selected features: 15
X shape: (5000, 15)
y shape: (5000,)


## 3.5 — Fix DAYS_EMPLOYED Anomaly

The value 365243 (≈1000 years) indicates unemployment.
Replace with NaN (missing) so median imputation handles it properly.

In [6]:
# Check how many rows have the anomalous value
anomalous_count = (X['DAYS_EMPLOYED'] == 365243).sum()
print(f"Anomalous DAYS_EMPLOYED values: {anomalous_count:,} ({anomalous_count/len(X):.1%} of data)")

# Replace anomaly with NaN
X['DAYS_EMPLOYED'] = X['DAYS_EMPLOYED'].replace(365243, np.nan)

print(f"✓ Replaced with NaN for median imputation")

Anomalous DAYS_EMPLOYED values: 0 (0.0% of data)
✓ Replaced with NaN for median imputation


## 3.6 — Convert Days to Years

Transform negative days into positive, interpretable years.
- Readability: -14,585 days → 39.9 years
- API consistency: Users provide age in years, not negative days

In [7]:
# DAYS_BIRTH: negative days before application
X['AGE_YEARS'] = (-X['DAYS_BIRTH'] / 365.25).round(1)

# DAYS_EMPLOYED: negative days worked (or NaN for unemployed)
X['YEARS_EMPLOYED'] = (-X['DAYS_EMPLOYED'] / 365.25).round(1)

# DAYS_ID_PUBLISH: years since ID was issued
X['YEARS_ID_PUBLISH'] = (-X['DAYS_ID_PUBLISH'] / 365.25).round(1)

# Drop raw DAYS columns
X = X.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH'])

print("✓ Converted days to years")
print(f"Age range: {X['AGE_YEARS'].min():.1f} - {X['AGE_YEARS'].max():.1f} years")
print(f"Employment range: {X['YEARS_EMPLOYED'].min():.1f} - {X['YEARS_EMPLOYED'].max():.1f} years")

✓ Converted days to years
Age range: 18.0 - 80.0 years
Employment range: 0.0 - 27.4 years


## 3.7 — Encode Categorical Variables

ML models only understand numbers. Convert categorical to numeric.

In [9]:
# Binary encoding: M/F → 0/1
X['CODE_GENDER'] = X['CODE_GENDER'].map({'M': 0, 'F': 1}).fillna(0).astype(int)

# Binary encoding: N/Y → 0/1
X['FLAG_OWN_CAR'] = X['FLAG_OWN_CAR'].map({'N': 0, 'Y': 1}).fillna(0).astype(int)
X['FLAG_OWN_REALTY'] = X['FLAG_OWN_REALTY'].map({'N': 0, 'Y': 1}).fillna(0).astype(int)

# Ordinal encoding for education (preserves ordering: 0 < 1 < 2 < 3 < 4)
education_map = {
    'Lower secondary': 0,
    'Secondary / secondary special': 1,
    'Incomplete higher': 2,
    'Higher education': 3,
    'Academic degree': 4,
}
X['EDUCATION_LEVEL'] = X['NAME_EDUCATION_TYPE'].map(education_map).fillna(1).astype(int)
X = X.drop(columns=['NAME_EDUCATION_TYPE'])

print("✓ Encoded categorical variables")

✓ Encoded categorical variables


## 3.8 — Handle Missing Values

In [10]:
# Check missing values
print("Missing values before filling:")
missing = X.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
    print(f"\nTotal missing: {missing.sum()} cells")
else:
    print("No missing values")

Missing values before filling:
No missing values


In [11]:
# Fill with median (robust to outliers)
X = X.fillna(X.median())

print(f"✓ Filled missing values with median")
print(f"Missing values after filling: {X.isnull().sum().sum()}")

✓ Filled missing values with median
Missing values after filling: 0


## 3.9 — Feature Engineering

Create three ratio features that capture financial burden.

- **CREDIT_INCOME_RATIO:** How many years of income is the loan?
- **ANNUITY_INCOME_RATIO:** What % of income goes to monthly payments?
- **CREDIT_GOODS_RATIO:** What % of purchase is financed?

In [ ]:
X['CREDIT_INCOME_RATIO'] = X['AMT_CREDIT'] / (X['AMT_INCOME_TOTAL'] + 1)
X['ANNUITY_INCOME_RATIO'] = X['AMT_ANNUITY'] / (X['AMT_INCOME_TOTAL'] + 1)
X['CREDIT_GOODS_RATIO'] = X['AMT_CREDIT'] / (X['AMT_GOODS_PRICE'] + 1)

print("✓ Engineered 3 ratio features")
print(f"\nFeature statistics:")
print(f"  CREDIT_INCOME_RATIO:    mean={X['CREDIT_INCOME_RATIO'].mean():.2f}, max={X['CREDIT_INCOME_RATIO'].max():.2f}")
print(f"  ANNUITY_INCOME_RATIO:   mean={X['ANNUITY_INCOME_RATIO'].mean():.2f}, max={X['ANNUITY_INCOME_RATIO'].max():.2f}")
print(f"  CREDIT_GOODS_RATIO:     mean={X['CREDIT_GOODS_RATIO'].mean():.2f}, max={X['CREDIT_GOODS_RATIO'].max():.2f}")

✓ Engineered 3 ratio features

Feature statistics:
  CREDIT_INCOME_RATIO:    mean=3.96, max=84.73
  ANNUITY_INCOME_RATIO:   mean=0.18, max=1.88
  CREDIT_GOODS_RATIO:     mean=1.12, max=6.00


## 3.10 — Verify Final Feature Set

In [ ]:
print(f"Final dataset shape: {X.shape}")
print(f"\nFeatures ({X.shape[1]} total):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nData types:")
print(X.dtypes)

print(f"\nMissing values: {X.isnull().sum().sum()}")
print(f"Duplicates: {X.duplicated().sum()}")

Final dataset shape: (307511, 18)

Features (18 total):
   1. EXT_SOURCE_1
   2. EXT_SOURCE_2
   3. EXT_SOURCE_3
   4. AMT_INCOME_TOTAL
   5. AMT_CREDIT
   6. AMT_ANNUITY
   7. AMT_GOODS_PRICE
   8. CODE_GENDER
   9. FLAG_OWN_CAR
  10. FLAG_OWN_REALTY
  11. CNT_CHILDREN
  12. AGE_YEARS
  13. YEARS_EMPLOYED
  14. YEARS_ID_PUBLISH
  15. EDUCATION_LEVEL
  16. CREDIT_INCOME_RATIO
  17. ANNUITY_INCOME_RATIO
  18. CREDIT_GOODS_RATIO

Data types:
EXT_SOURCE_1            float64
EXT_SOURCE_2            float64
EXT_SOURCE_3            float64
AMT_INCOME_TOTAL        float64
AMT_CREDIT              float64
AMT_ANNUITY             float64
AMT_GOODS_PRICE         float64
CODE_GENDER               int64
FLAG_OWN_CAR              int64
FLAG_OWN_REALTY           int64
CNT_CHILDREN              int64
AGE_YEARS               float64
YEARS_EMPLOYED          float64
YEARS_ID_PUBLISH        float64
EDUCATION_LEVEL           int64
CREDIT_INCOME_RATIO     float64
ANNUITY_INCOME_RATIO    float64
CREDIT_GOODS

In [ ]:
# Save for model training
# These will be split into train/test in the next notebook
print("\n✓ Data preparation complete!")
print(f"Ready for model training: X={X.shape}, y={y.shape}")


✓ Data preparation complete!
Ready for model training: X=(307511, 18), y=(307511,)
